In [5]:
# 1. You must explicitly enable the experimental features first
from sklearn.experimental import enable_iterative_imputer

# 2. Now you can import IterativeImputer normally
from sklearn.impute import IterativeImputer

In [6]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.experimental import enable_iterative_imputer
from sklearn.experimental import enable_iterative_imputer

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

from category_encoders import TargetEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from scipy.stats import rankdata

# =========================================================
# 1. LOAD DATA
# =========================================================

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

TARGET = "Drafted"

y = train[TARGET]
train_ids = train["Id"]
test_ids = test["Id"]

# =========================================================
# 2. FEATURE ENGINEERING
# =========================================================

physical_stats = [
    'Sprint_40yd',
    'Vertical_Jump',
    'Weight',
    'Height',
    'Broad_Jump'
]

for df in [train, test]:

    # -----------------------------------------------------
    # Basic Features
    # -----------------------------------------------------

    df['BMI'] = df['Weight'] / (df['Height'] ** 2 + 1e-6)

    df['Explosiveness'] = (
        df['Vertical_Jump'] +
        df['Broad_Jump']
    )

    # -----------------------------------------------------
    # Athletic Features
    # -----------------------------------------------------

    df['Height_Weight'] = (
        df['Height'] * df['Weight']
    )

    df['Speed_Score'] = (
        df['Weight'] *
        (200 / (df['Sprint_40yd'] ** 4 + 1e-6))
    )

    df['Agility_BMI'] = (
        df['BMI'] /
        (df['Sprint_40yd'] + 1e-6)
    )

    df['Explosion_Ratio'] = (
        df['Explosiveness'] /
        (df['Weight'] + 1)
    )

    df['Height_to_Weight'] = (
        df['Height'] /
        (df['Weight'] + 1)
    )

    df['Athleticism_Index'] = (
        df['Vertical_Jump'] *
        df['Broad_Jump'] /
        (df['Sprint_40yd'] + 1)
    )

    # -----------------------------------------------------
    # Position Relative Features
    # -----------------------------------------------------

    for stat in physical_stats:

        pos_mean = df.groupby('Position')[stat].transform('mean')

        df[f'{stat}_pos_rel'] = (
            df[stat] - pos_mean
        )

        df[f'{stat}_rank'] = (
            df.groupby('Position')[stat]
            .rank(pct=True)
        )

# =========================================================
# 3. SCHOOL FREQUENCY ENCODING
# =========================================================

school_freq = train['School'].value_counts()

train['School_Freq'] = (
    train['School']
    .map(school_freq)
)

test['School_Freq'] = (
    test['School']
    .map(school_freq)
    .fillna(0)
)

# =========================================================
# 4. DROP UNUSED COLUMNS
# =========================================================

drop_cols = ["Id", TARGET]

X = train.drop(columns=drop_cols)
test = test.drop(columns=["Id"])

# =========================================================
# 5. TARGET ENCODING
# =========================================================

cat_cols = [
    'Player_Type',
    'Position_Type',
    'Position',
    'School'
]

te = TargetEncoder(cols=cat_cols)

X[cat_cols] = te.fit_transform(X[cat_cols], y)
test[cat_cols] = te.transform(test[cat_cols])

# =========================================================
# 6. IMPUTATION
# =========================================================

imputer = IterativeImputer(
    random_state=42,
    max_iter=20
)

X = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)

test = pd.DataFrame(
    imputer.transform(test),
    columns=test.columns
)

# =========================================================
# 7. SCALING
# =========================================================

scaler = RobustScaler()

X = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

test = pd.DataFrame(
    scaler.transform(test),
    columns=test.columns
)

# =========================================================
# 8. MODELS
# =========================================================

cat_params = {
    'iterations': 3000,
    'depth': 5,
    'learning_rate': 0.01,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'l2_leaf_reg': 5,
    'subsample': 0.8,
    'random_seed': 42,
    'verbose': 0
}

xgb_params = {
    'n_estimators': 2500,
    'max_depth': 4,
    'learning_rate': 0.01,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma': 1,
    'reg_alpha': 0.5,
    'reg_lambda': 2,
    'eval_metric': 'auc',
    'random_state': 42,
    'use_label_encoder': False
}

lgbm_params = {
    'n_estimators': 2500,
    'learning_rate': 0.01,
    'num_leaves': 31,
    'max_depth': -1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 1,
    'reg_lambda': 2,
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'random_state': 42
}

models = [
    ("cat", CatBoostClassifier(**cat_params)),
    ("xgb", XGBClassifier(**xgb_params)),
    ("lgbm", LGBMClassifier(**lgbm_params))
]

# =========================================================
# 9. CROSS VALIDATION
# =========================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros((len(X), len(models)))
test_preds = np.zeros((len(test), len(models)))

# =========================================================
# 10. TRAIN BASE MODELS
# =========================================================

for model_idx, (name, model) in enumerate(models):

    print(f"\nTraining {name.upper()}")

    fold_test_preds = np.zeros((len(test), 10))

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

        print(f"Fold {fold+1}")

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        # -------------------------------------------------
        # FIT MODEL
        # -------------------------------------------------

        if name == "cat":

            model.fit(
                X_train,
                y_train,
                eval_set=(X_valid, y_valid),
                use_best_model=True,
                early_stopping_rounds=100,
                verbose=False
            )

        elif name == "xgb":

            model.fit(
                X_train,
                y_train,
                eval_set=[(X_valid, y_valid)],
                verbose=False
            )

        else:

            model.fit(
                X_train,
                y_train,
                eval_set=[(X_valid, y_valid)],
                callbacks=[]
            )

        # -------------------------------------------------
        # PREDICTIONS
        # -------------------------------------------------

        val_pred = model.predict_proba(X_valid)[:, 1]
        test_pred = model.predict_proba(test)[:, 1]

        oof_preds[valid_idx, model_idx] = val_pred
        fold_test_preds[:, fold] = test_pred

        fold_auc = roc_auc_score(y_valid, val_pred)

        print(f"AUC: {fold_auc:.5f}")

    test_preds[:, model_idx] = fold_test_preds.mean(axis=1)

# =========================================================
# 11. BASE MODEL AUC
# =========================================================

for i, (name, _) in enumerate(models):

    auc = roc_auc_score(y, oof_preds[:, i])

    print(f"{name.upper()} OOF AUC: {auc:.5f}")

# =========================================================
# 12. META FEATURES
# =========================================================

meta_train = np.concatenate(
    [oof_preds, X.values],
    axis=1
)

meta_test = np.concatenate(
    [test_preds, test.values],
    axis=1
)

# =========================================================
# 13. LOGISTIC REGRESSION STACKER
# =========================================================

meta_model = LogisticRegression(
    C=0.5,
    max_iter=5000,
    random_state=42
)

meta_model.fit(meta_train, y)

stack_preds = meta_model.predict_proba(meta_test)[:, 1]

# =========================================================
# 14. RANK AVERAGING
# =========================================================

rank_avg_preds = (
    rankdata(test_preds[:, 0]) +
    rankdata(test_preds[:, 1]) +
    rankdata(test_preds[:, 2])
) / 3

rank_avg_preds = rank_avg_preds / rank_avg_preds.max()

# =========================================================
# 15. FINAL BLEND
# =========================================================

final_predictions = (
    0.7 * stack_preds +
    0.3 * rank_avg_preds
)

# =========================================================
# 16. SAVE SUBMISSION
# =========================================================

submission = sample_sub.copy()

submission["Drafted"] = final_predictions

submission.to_csv(
    "submission_high_auc.csv",
    index=False
)

print("\nSubmission file saved:")
print("submission_high_auc.csv")


Training CAT
Fold 1
AUC: 0.83318
Fold 2
AUC: 0.85396
Fold 3
AUC: 0.88204
Fold 4
AUC: 0.88084
Fold 5
AUC: 0.86967
Fold 6
AUC: 0.88163
Fold 7
AUC: 0.80204
Fold 8
AUC: 0.82455
Fold 9
AUC: 0.83611
Fold 10
AUC: 0.88804

Training XGB
Fold 1
AUC: 0.81559
Fold 2
AUC: 0.84741
Fold 3
AUC: 0.87002
Fold 4
AUC: 0.85510
Fold 5
AUC: 0.83441
Fold 6
AUC: 0.86310
Fold 7
AUC: 0.78141
Fold 8
AUC: 0.81967
Fold 9
AUC: 0.81837
Fold 10
AUC: 0.86383

Training LGBM
Fold 1
AUC: 0.82050
Fold 2
AUC: 0.84006
Fold 3
AUC: 0.88609
Fold 4
AUC: 0.85357
Fold 5
AUC: 0.82557
Fold 6
AUC: 0.84246
Fold 7
AUC: 0.77080
Fold 8
AUC: 0.81899
Fold 9
AUC: 0.80833
Fold 10
AUC: 0.87630
CAT OOF AUC: 0.84937
XGB OOF AUC: 0.83667
LGBM OOF AUC: 0.83351

Submission file saved:
submission_high_auc.csv
